# Preparing Data for the Algorithm

## 1. Load all data

In [1]:
# Manejo de datos y operaciones matemáticas
import pandas as pd
import numpy as np

# Visualización de datos
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid") # Configuración estética de Seaborn

# Procesamiento de imágenes/vídeo
import cv2

# Lectura de metadatos y archivos serializados
import json
import pickle
import os, sys # Útil para manejar las rutas 

# Para cargar datos
from kloppy import sportscode, metrica
sys.path.append('..')
import games

In [2]:
# Load Tagging
sys.path.insert(0, os.path.abspath('..'))
print("Cargando Smart Tagging (Eventos)...")
path = games.asset_paths()['smart_tagging']
events_df = sportscode.load(path).to_df()
print(f"-> {len(events_df)} instancias cargadas con éxito.")
display(events_df.head(20))

Cargando Smart Tagging (Eventos)...
-> 638 instancias cargadas con éxito.


,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,None,None
1,2,1,0 days 00:33:05.240000,0 days 00:33:10.240000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
2,3,1,0 days 00:41:20.480000,0 days 00:41:25.480000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
3,4,1,0 days 00:50:18.280000,0 days 00:50:23.280000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
4,5,1,0 days 01:15:46.400000,0 days 01:15:51.400000,SET PIECES,N/A,Kick Off (Start),2nd Half,None,None,None
5,6,1,0 days 01:48:40,0 days 01:48:45,SET PIECES,N/A,Kick Off (After Goal),2nd Half,None,None,None
6,7,1,0 days 01:17:41.720000,0 days 01:18:09.400000,PLAYERS IN THE BOX,Barcelona,None,2nd Half,4,None,None
7,8,1,0 days 01:18:42.760000,0 days 01:19:07.040000,PLAYERS IN THE BOX,Barcelona,None,2nd Half,4,None,None
8,9,1,0 days 01:21:23.200000,0 days 01:21:44.600000,PLAYERS IN THE BOX,Barcelona,None,2nd Half,2,None,None
9,10,1,0 days 01:22:46.080000,0 days 01:23:11.160000,PLAYERS IN THE BOX,Barcelona,None,2nd Half,2,None,None


In [3]:
# Load ATD
sys.path.insert(0, os.path.abspath('..'))

print("Cargando Tracking ATD...")
paths = games.asset_paths(source='tactical')

dataset = metrica.load_tracking_epts(
    meta_data=paths['atd_meta'],
    raw_data=paths['atd_raw'], 
)

print(f"-> {len(dataset.records)} frames cargados con éxito.")
frame_df = dataset.to_df()
display(frame_df.head())

Cargando Tracking ATD...
-> 199319 frames cargados con éxito.


,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,1_x,...,44_d,44_s,45_x,45_y,45_d,45_s,46_x,46_y,46_d,46_s
0,1,0 days 00:00:00.040000,1,None,None,NaN,NaN,None,None,0.33411,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
1,1,0 days 00:00:00.080000,2,None,None,NaN,NaN,None,None,0.33404,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
2,1,0 days 00:00:00.120000,3,None,None,NaN,NaN,None,None,0.33396,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
3,1,0 days 00:00:00.160000,4,None,None,NaN,NaN,None,None,0.33389,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
4,1,0 days 00:00:00.200000,5,None,None,NaN,NaN,None,None,0.33382,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN


## 2. Structure Dataset

In [18]:
import pandas as pd
import numpy as np

def build_hackathon_dataset_by_code(tracking_df, events_df, fps=25):
    # 1. Ordenar el tracking cronológicamente por frame_id
    df = tracking_df.sort_values('frame_id').reset_index(drop=True)
    
    # 2. Asegurar cálculo de frames de inicio y fin desde timedelta
    if 'start_frame' not in events_df.columns:
        if pd.api.types.is_timedelta64_dtype(events_df['timestamp']):
            start_seconds = events_df['timestamp'].dt.total_seconds()
        else:
            start_seconds = pd.to_numeric(events_df['timestamp'], errors='coerce')
        events_df['start_frame'] = (start_seconds * fps).astype(int)
        
    if 'end_frame' not in events_df.columns:
        if pd.api.types.is_timedelta64_dtype(events_df['end_timestamp']):
            end_seconds = events_df['end_timestamp'].dt.total_seconds()
        else:
            end_seconds = pd.to_numeric(events_df['end_timestamp'], errors='coerce')
        events_df['end_frame'] = (end_seconds * fps).astype(int)
        
    # --- DIAGNÓSTICO RÁPIDO PARA VER SI COINCIDEN ---
    print(f"Rango de frame_id en tracking: {df['frame_id'].min()} a {df['frame_id'].max()}")
    print(f"Rango de start_frame en events: {events_df['start_frame'].min()} a {events_df['start_frame'].max()}")
    # -----------------------------------------------

    unique_codes = events_df['code'].dropna().unique()
    print(unique_codes )
    
    for code in unique_codes:
        clean_code = str(code).strip().lower().replace(' ', '_').replace('-', '_')
        
        col_value = f"{clean_code}"
        col_team = f"{clean_code}_team"
        
        # IMPORTANTE: Inicializamos como object/None para aceptar texto sin warnings
        df[col_value] = None
        df[col_team] = None
        
        sub_events = events_df[events_df['code'] == code]
        
        for _, ev in sub_events.iterrows():
            start = ev['start_frame']
            end = ev['end_frame']
            
            # Condicional para set pieces vs el resto
            if 'PLAYERS IN THE BOX' in code:
                val = ev.get('Max Players in the box', None)
            elif 'SET PIECES' in code:
                val = ev.get('Type', None)
            else:
                val = True  # Indicador booleano de presencia para el resto de eventos
                
            team = ev.get('Team', None)
            
            # Máscara de los frames
            mask = (df['frame_id'] >= start) & (df['frame_id'] <= end)
            
            df.loc[mask, col_value] = val
            df.loc[mask, col_team] = team
            
    return df

In [19]:
# Ejecutamos la función (asumiendo que el framerate de tu vídeo es de 25 fps, cámbialo si es 50fps)
df_hackathon = build_hackathon_dataset_by_code(frame_df, events_df, fps=25)

Rango de frame_id en tracking: 1 a 199319
Rango de start_frame en events: 11069 a 191965
['SET PIECES' 'PLAYERS IN THE BOX' 'BALL IN FINAL THIRD' 'BALL IN THE BOX'
 'GOALS' 'PROGRESSION' 'ATTACKING TRANSITION' 'BUILD UP' 'LONG BALL'
 'CREATING CHANCES' 'DEFENDING IN MIDDLE THIRD' 'DEFENSIVE TRANSITION'
 'DEFENDING IN ATTACKING THIRD' 'DEFENDING IN DEFENSIVE THIRD']


In [25]:
# 1. Definimos las columnas específicas que quieres conservar además de las que terminan en '_d'
cols_especificas = [
    'ball_state', 
    'ball_owning_team_id', 
    'ball_z', 
    'set_pieces_team'
]

# 2. Buscamos todas las columnas del DataFrame que terminan en '_d'
cols_terminan_en_d = [col for col in df_hackathon.columns if col.endswith('_d')]

# 3. Unimos ambas listas asegurándonos de que no haya duplicados (usando set) y mantenemos el orden si es necesario
# O directamente filtramos el DataFrame combinándolas de forma segura:
cols_a_borrar = list(set(cols_terminan_en_d + [c for c in cols_especificas if c in df_hackathon.columns]))

# 4. Creamos el nuevo DataFrame filtrado
df_hackathon = df_hackathon.drop(columns=cols_a_borrar)

In [29]:
# 3. Ejecútalo de nuevo
# 1. Creamos una máscara para filtrar donde 'Side' o 'Direction of ball entry' no sean nulos
mask = events_df['Side'].notna() | events_df['Direction of ball entry'].notna()

# 2. Aplicamos el filtro al DataFrame de eventos
events_filtrados = events_df[mask]

# 3. Mostramos el resultado (puedes usar display o print según prefieras)
display(events_filtrados)

,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry,start_frame,end_frame
103,104,1,0 days 00:07:22.760000,0 days 00:08:33.600000,BALL IN FINAL THIRD,Atletic Club,None,1st Half,4,Central,None,11069,12840
104,105,1,0 days 00:11:21.160000,0 days 00:11:48.240000,BALL IN FINAL THIRD,Atletic Club,None,1st Half,4,Left,None,17029,17706
105,106,1,0 days 00:11:43.040000,0 days 00:12:11,BALL IN FINAL THIRD,Atletic Club,None,1st Half,6,Left,None,17576,18275
106,107,1,0 days 00:15:03.760000,0 days 00:15:40.640000,BALL IN FINAL THIRD,Atletic Club,None,1st Half,3,Central,None,22594,23516
107,108,1,0 days 00:15:42.360000,0 days 00:16:03.240000,BALL IN FINAL THIRD,Atletic Club,None,1st Half,4,Central,None,23559,24081
108,109,1,0 days 00:23:55.720000,0 days 00:24:20.560000,BALL IN FINAL THIRD,Atletic Club,None,1st Half,4,Right,None,35893,36514
109,110,1,0 days 00:24:32.520000,0 days 00:25:21.120000,BALL IN FINAL THIRD,Atletic Club,None,1st Half,7+,Central,None,36813,38028
110,111,1,0 days 00:27:29.600000,0 days 00:28:28,BALL IN FINAL THIRD,Atletic Club,None,1st Half,5,Left,None,41240,42700
111,112,1,0 days 00:28:19.960000,0 days 00:28:46.520000,BALL IN FINAL THIRD,Atletic Club,None,1st Half,5,Central,None,42499,43163
112,113,1,0 days 00:33:13.360000,0 days 00:33:40.480000,BALL IN FINAL THIRD,Atletic Club,None,1st Half,4,Central,None,49834,50512


## 3. View Dataset Caractheristics

In [28]:
import pandas as pd

# 1. Ampliamos los límites de visualización para que no oculte filas de la tabla
pd.set_option('display.max_rows', None)

# 2. Volvemos a definir la función (o úsala tal cual la tienes)
def inspect_hackathon_dataset(df):
    print("=== 1. DIMENSIONES DEL DATASET ===" )
    print(f"Total de frames (filas): {df.shape[0]:,}")
    print(f"Total de columnas: {df.shape[1]}")
    
    print("\n=== 2. TIPOS DE DATOS Y VALORES NULOS ===")
    info_df = pd.DataFrame({
        'Tipo de Dato': df.dtypes,
        'Nulos': df.isnull().sum(),
        '% Nulos': (df.isnull().mean() * 100).round(2)
    })
    # Ahora mostrará toda la lista completa de columnas sin truncar el medio
    display(info_df)
    
    print("\n=== 3. RESUMEN ESTADÍSTICO (VARIABLES NUMÉRICAS) ===")
    display(df.describe())
    
    print("\n=== 4. ESTADÍSTICAS DE VARIABLES CATEGÓRICAS Y EVENTOS ===")
    cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    display(df[cat_cols].describe())

# 3. Ejecútalo de nuevo
inspect_hackathon_dataset(df_hackathon)

=== 1. DIMENSIONES DEL DATASET ===
Total de frames (filas): 199,319
Total de columnas: 171

=== 2. TIPOS DE DATOS Y VALORES NULOS ===


,Tipo de Dato,Nulos,% Nulos
period_id,int64,0,0.00
timestamp,timedelta64[ns],0,0.00
frame_id,int64,0,0.00
ball_x,float64,85831,43.06
ball_y,float64,85831,43.06
ball_speed,object,199319,100.00
1_x,float64,119645,60.03
1_y,float64,119645,60.03
1_s,float64,123338,61.88
2_x,float64,121211,60.81



=== 3. RESUMEN ESTADÍSTICO (VARIABLES NUMÉRICAS) ===


,period_id,timestamp,frame_id,ball_x,ball_y,1_x,1_y,1_s,2_x,2_y,...,43_s,44_x,44_y,44_s,45_x,45_y,45_s,46_x,46_y,46_s
count,199319.0,199319,199319.00000,113488.000000,113488.000000,79674.000000,79674.000000,75981.000000,78108.000000,78108.000000,...,84186.000000,0.0,0.0,0.0,121292.000000,121292.000000,117868.000000,178926.00000,178926.000000,67784.000000
mean,1.0,0 days 01:06:26.400000,99660.00000,0.502648,0.570560,0.473240,0.536663,1.875471,0.495088,0.529493,...,1.644408,NaN,NaN,NaN,0.492229,0.519714,1.795823,0.51952,0.492111,0.984424
std,0.0,0 days 00:38:21.543339587,57538.58349,0.239323,0.284386,0.193351,0.198242,1.507069,0.211607,0.216892,...,1.342463,NaN,NaN,NaN,0.213959,0.199299,1.414617,0.43113,0.066053,0.835118
min,1.0,0 days 00:00:00.040000,1.00000,-0.065170,-0.107130,0.002240,0.007940,0.002636,0.003060,-0.036540,...,0.002068,NaN,NaN,NaN,-0.035360,-0.035400,0.000339,-0.03187,0.216570,0.000419
25%,1.0,0 days 00:33:13.220000,49830.50000,0.318747,0.356928,0.337990,0.406223,0.890337,0.360470,0.403097,...,0.740832,NaN,NaN,NaN,0.352845,0.391967,0.813838,0.06235,0.466800,0.376709
50%,1.0,0 days 01:06:26.400000,99660.00000,0.529655,0.585035,0.468540,0.514260,1.448702,0.479445,0.525660,...,1.279403,NaN,NaN,NaN,0.501810,0.507680,1.378205,0.84086,0.500440,0.841198
75%,1.0,0 days 01:39:39.580000,149489.50000,0.675230,0.804560,0.605067,0.651310,2.573503,0.663420,0.683490,...,2.169647,NaN,NaN,NaN,0.628282,0.651990,2.583421,0.93661,0.519390,1.329495
max,1.0,0 days 02:12:52.760000,199319.00000,1.113850,1.488540,0.966180,1.071450,11.293826,1.043250,1.073310,...,9.070786,NaN,NaN,NaN,1.017030,1.053480,8.491470,1.01078,0.770850,6.930494



=== 4. ESTADÍSTICAS DE VARIABLES CATEGÓRICAS Y EVENTOS ===


,ball_speed,set_pieces,players_in_the_box,players_in_the_box_team,ball_in_final_third,ball_in_final_third_team,ball_in_the_box,ball_in_the_box_team,goals,goals_team,...,creating_chances,creating_chances_team,defending_in_middle_third,defending_in_middle_third_team,defensive_transition,defensive_transition_team,defending_in_attacking_third,defending_in_attacking_third_team,defending_in_defensive_third,defending_in_defensive_third_team
count,0,44037,81389,81389,61343,61343,29755,29755,16507,16507,...,22413,22413,31189,31189,9134,9134,11033,11033,22413,22413
unique,0,6,7,2,1,2,1,2,1,2,...,1,3,1,2,1,2,1,2,1,3
top,NaN,Throw-in,1,Barcelona,True,Barcelona,True,Barcelona,True,Barcelona,...,True,Barcelona,True,Atletic Club,True,Atletic Club,True,Atletic Club,True,Atletic Club
freq,NaN,21000,14947,46197,61343,36165,29755,16718,16507,9979,...,22413,14269,31189,23119,9134,5031,11033,7814,22413,14269


## 4. Save df

In [ ]:
import os
import pandas as pd

# 1. Definir la ruta de la carpeta y el nombre del archivo
carpeta_destino = 'data'
nombre_archivo = 'dataframe_hackaton_final.csv'

# 2. Crear la carpeta 'data' si no existe
os.makedirs(carpeta_destino, exist_ok=True)

# 3. Construir la ruta completa del archivo
ruta_completa = os.path.join(carpeta_destino, nombre_archivo)

# 4. Guardar el DataFrame en CSV
# index=False evita que Pandas guarde el índice numérico como una columna extra
df_hackathon.to_csv(ruta_completa, index=False)

print(f"¡DataFrame guardado con éxito en: {ruta_completa}")